# 07b — embryo floor plate subclustering

**Feeds:** Fig 4d, 4e (via stage 12)

**Position in the chain:** parity lineage. Supplies the embryo Floor Plate label the 18-label merge needs.

Ported from the original analysis. Saved cell outputs are from the original run, and paths in them appear as `<analysis-root>/...`.

This notebook ran on the earlier human embryo clustering object (`parity/results/intermediates/07_human_embryo/adata_embryo_with_clusters.h5ad`), and the SMD z-scores shipped for it come from that run, so it does not run on the embryo object `07_human_embryo.ipynb` builds here. The earlier object is in the Zenodo deposit (`../../data/DOWNLOAD.md`); `scrnaseq/run_chain.py` runs this notebook when it is present under `SCRNASEQ_INPUT_ROOT`, and skips it otherwise.

**Changes from the original notebook**, so that it runs from this repository:
1. Paths. The first code cell finds the repository from any working directory inside it; the original looked for `src/` at most two levels up, in the `parity` analysis directory. `PROJECT_ROOT` is `scrnaseq/chain_inputs/`, the shipped files that no notebook writes, laid out as the original analysis directory. The notebook reads and writes under `DEV_ROOT`, `$SCRNASEQ_RESULTS_ROOT/parity/` (default `scrnaseq/output/parity/`), instead of the `parity` directory itself.
2. The SMD z-scores for the embryo floor plate (IVSC) subset are read from `scrnaseq/chain_inputs/parity/results/intermediates/07b_human_embryo_floor_plate_subclustering/` instead of the stage's output directory.

3. The embryo clusters object. `SCRNASEQ_INPUT_ROOT` (default `data/scrnaseq_inputs/`) is checked for `earlier_embryo_run/results/intermediates/07_human_embryo/adata_embryo_with_clusters.h5ad`; when absent, the notebook falls back to `embryo_stage_path`, as before this override existed (in practice unreachable, since `run_chain.py` only runs this notebook when the object is present).

No other line of code was changed.


# 07b - Human Embryo Floor Plate Subclustering

Note: this stage is currently parity-only and should be upgraded into trunk_main_dev in a future pass.


Depends on: parity `07_human_embryo` outputs and returned SMD z-scores for the IVSC subset.

In [ ]:

from pathlib import Path
import os
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "trunk_morph_ref").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root containing src/trunk_morph_ref/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trunk_morph_ref.paths import chain_inputs_root, scrnaseq_input_root

# Inputs that no notebook writes: scrnaseq/chain_inputs/, laid out as the original analysis directory.
PROJECT_ROOT = chain_inputs_root(REPO_ROOT)
# The earlier human embryo clusters object, fetched from the Zenodo deposit (see data/DOWNLOAD.md).
SCRNASEQ_INPUT_ROOT = scrnaseq_input_root(REPO_ROOT)

os.chdir(REPO_ROOT)
os.environ.pop("NUMBA_DISABLE_JIT", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")

import numba

_old_vectorize = numba.vectorize
_old_jit = numba.jit
_old_njit = numba.njit

def _force_no_cache(kwargs):
    kwargs = dict(kwargs)
    kwargs["cache"] = False
    return kwargs

def vectorize_no_cache(*args, **kwargs):
    return _old_vectorize(*args, **_force_no_cache(kwargs))

def jit_no_cache(*args, **kwargs):
    return _old_jit(*args, **_force_no_cache(kwargs))

def njit_no_cache(*args, **kwargs):
    return _old_njit(*args, **_force_no_cache(kwargs))

numba.vectorize = vectorize_no_cache
numba.jit = jit_no_cache
numba.njit = njit_no_cache


## Setup and Imports

In [ ]:

import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import seaborn as sns
from cmcrameri.cm import batlow
from scipy.sparse import save_npz


In [ ]:
plt.rcParams["svg.fonttype"] = "none"  # Keep fonts editable in Illustrator

In [ ]:

from src.trunk_morph_ref.cell_ordering import ordered_cells_by_cluster_clustermap
from src.trunk_morph_ref.correlation import (
    gene_corrcoef_dense_legacy as gene_corrcoef_sparse_safe,
)
from src.trunk_morph_ref.paths import init_output_paths, scrnaseq_results_root
from src.trunk_morph_ref.pipeline_io import load_h5ad, save_h5ad, save_json, stage_dir
from src.trunk_morph_ref.preprocessing import (
    assert_gene_id_index,
    ensure_gene_id_index,
    filter_genes_dense_legacy as filter_genes,
    install_scanpy_symbol_defaults,
    normalize_unit_variance_dense_legacy as normalize_unit_variance,
    resolve_symbols,
    var_names_to_symbols,
)

# Outputs: $SCRNASEQ_RESULTS_ROOT (default scrnaseq/output/), under parity/ as in the original.
DEV_ROOT = scrnaseq_results_root(REPO_ROOT) / "parity"
RESULTS_DIR, MANUSCRIPT_FIG_DIR, EXTENDED_FIG_DIR = init_output_paths(DEV_ROOT)
install_scanpy_symbol_defaults(sc)


In [ ]:

STAGE_ID = "07b_human_embryo_floor_plate_subclustering"
stage_path = stage_dir(RESULTS_DIR, STAGE_ID)
fig_path = stage_path / "figures"
table_path = stage_path / "tables"
stage_path.mkdir(parents=True, exist_ok=True)
fig_path.mkdir(parents=True, exist_ok=True)
table_path.mkdir(parents=True, exist_ok=True)

embryo_stage_path = stage_dir(RESULTS_DIR, "07_human_embryo")

# This notebook ran on the earlier human embryo clustering object; nothing in this chain writes
# it to embryo_stage_path (the parity lineage is not rebuilt here). Read it from the Zenodo
# deposit when present, otherwise fall back to embryo_stage_path, as before this override existed.
_earlier_embryo_clusters_path = (
    SCRNASEQ_INPUT_ROOT
    / "earlier_embryo_run/results/intermediates/07_human_embryo/adata_embryo_with_clusters.h5ad"
)
if _earlier_embryo_clusters_path.exists():
    adata_embryo_with_clusters = load_h5ad(_earlier_embryo_clusters_path)
else:
    adata_embryo_with_clusters = load_h5ad(embryo_stage_path / "adata_embryo_with_clusters.h5ad")
ensure_gene_id_index(adata_embryo_with_clusters)
assert_gene_id_index(adata_embryo_with_clusters)

print("Loaded upstream embryo artifact:")
print(embryo_stage_path / "adata_embryo_with_clusters.h5ad")
print(adata_embryo_with_clusters)


## Provenance Note

This notebook stays strictly downstream of parity `07_human_embryo`.

It does **not** recluster the full embryo dataset. Instead it:
- loads the staged parity `07` embryo object with saved cluster labels
- defines the parent cluster `Week 4 Intermediate-Ventral Spinal Cord`
- rebuilds a fresh downstream IVSC subset artifact from that stage-07 object
- reruns subset-level preprocessing for the IVSC subset alone
- loads the returned IVSC SMD z-scores
- screens a small Leiden resolution ladder in IVSC SMD gene space
- chooses the first raw Leiden resolution that isolates a single strong floor-plate-like block
- collapses that raw block against the remainder of IVSC
- shows the standard downstream intermediates for that floor-plate-vs-rest separation


In [ ]:

FP_POSITIVE_MARKERS = ["FOXA2", "SHH", "ARX", "CORIN", "PTCH1", "SOX2"]
FP_CONTEXT_MARKERS = ["NKX6-1", "NKX6-2", "FOXA1", "TFF3"]
NOTOCHORD_COUNTER_MARKERS = ["NOTO", "PKD1L1", "TFF3"]


def expr_vector_by_symbol(adata, symbol):
    gene_ids = resolve_symbols(adata, [symbol], strict=False, allow_missing=True)
    if not gene_ids:
        return None
    x = adata[:, gene_ids[0]].X
    if hasattr(x, "toarray"):
        x = x.toarray()
    return np.asarray(x).reshape(-1)


def cluster_marker_summary(adata_expr, cluster_key, marker_symbols, marker_cache=None):
    if marker_cache is None:
        marker_cache = {symbol: expr_vector_by_symbol(adata_expr, symbol) for symbol in marker_symbols}
    rows = []
    for cluster in adata_expr.obs[cluster_key].cat.categories:
        mask = (adata_expr.obs[cluster_key] == cluster).values
        row = {"cluster": cluster, "n_cells": int(mask.sum())}
        for symbol in marker_symbols:
            values = marker_cache.get(symbol)
            if values is None:
                row[f"{symbol}_mean"] = np.nan
                row[f"{symbol}_pct_gt1"] = np.nan
            else:
                row[f"{symbol}_mean"] = float(values[mask].mean())
                row[f"{symbol}_pct_gt1"] = float((values[mask] > 1).mean())
        rows.append(row)
    return pd.DataFrame(rows)


def floor_plate_score(marker_summary_df):
    return (
        marker_summary_df[[f"{g}_mean" for g in FP_POSITIVE_MARKERS]].fillna(0).sum(axis=1)
        - marker_summary_df[[f"{g}_mean" for g in NOTOCHORD_COUNTER_MARKERS]].fillna(0).sum(axis=1)
    )


## Prepare IVSC Parent-Cluster Subset

In [ ]:

ivsc_parent_clusters = ["Week 4 Intermediate-Ventral Spinal Cord"]

adata_embryo_ivsc = adata_embryo_with_clusters[
    adata_embryo_with_clusters.obs["leiden_embryo"].isin(ivsc_parent_clusters).values,
    :,
].copy()

print(f"Selected {adata_embryo_ivsc.n_obs} IVSC parent-cluster cells")
print(adata_embryo_ivsc.obs["leiden_embryo"].astype(str).value_counts())


## Reprocess IVSC Subset for SMD Input

In [ ]:

adata_embryo_ivsc_filtered = filter_genes(
    adata_embryo_ivsc,
    mean_cutoff=0.05,
    std_cutoff=0.05,
)
adata_embryo_ivsc_filtered = normalize_unit_variance(adata_embryo_ivsc_filtered)

counts = adata_embryo_ivsc_filtered.X
cv_mask = np.mean(counts, axis=0).transpose() < 1
adata_embryo_ivsc_filtered = adata_embryo_ivsc_filtered[:, cv_mask].copy()

print(
    f"{adata_embryo_ivsc_filtered.n_obs} cells after IVSC subset preprocessing\n"
    f"{adata_embryo_ivsc_filtered.n_vars} genes after preprocessing\n"
    f"Recommended n_sub ≈ {int(0.8 * adata_embryo_ivsc_filtered.n_obs)}"
)


In [ ]:

adata_embryo_ivsc_ = filter_genes(adata_embryo_ivsc.copy(), mean_cutoff=0, std_cutoff=0)
adata_embryo_ivsc_ = normalize_unit_variance(adata_embryo_ivsc_)

for _adata in [adata_embryo_with_clusters, adata_embryo_ivsc_filtered, adata_embryo_ivsc_]:
    ensure_gene_id_index(_adata)
    assert_gene_id_index(_adata)


## Load Returned IVSC SMD Z-Scores

In [ ]:

z_score_file = PROJECT_ROOT / "parity" / "results" / "intermediates" / STAGE_ID / "z_embryo_ivsc_07b_001.npy"
z_scores_embryo_ivsc = np.load(z_score_file)
if len(z_scores_embryo_ivsc) != adata_embryo_ivsc_filtered.n_vars:
    raise ValueError(
        f"{z_score_file.name} length ({len(z_scores_embryo_ivsc)}) does not match "
        f"filtered IVSC gene count ({adata_embryo_ivsc_filtered.n_vars})."
    )

adata_embryo_ivsc_filtered.var["z_score_embryo_ivsc"] = z_scores_embryo_ivsc
adata_embryo_ivsc_filtered.var["log1p_z_score_embryo_ivsc"] = np.log1p(z_scores_embryo_ivsc)
adata_embryo_ivsc_.var["z_score_embryo_ivsc"] = np.nan
for gene_id in adata_embryo_ivsc_filtered.var_names:
    adata_embryo_ivsc_.var.loc[gene_id, "z_score_embryo_ivsc"] = adata_embryo_ivsc_filtered.var.loc[
        gene_id, "z_score_embryo_ivsc"
    ]

print(f"Loaded {z_score_file.name} with {len(z_scores_embryo_ivsc)} IVSC z-scores")


In [ ]:

smd_cutoff = 2

plt.figure(figsize=(12, 4))
plt.hlines(smd_cutoff, -1, 2000, "r")
plt.plot(sorted(z_scores_embryo_ivsc)[::-1], "k.", markersize=5)
plt.yscale("log")
plt.ylim(0.01, 1.5 * z_scores_embryo_ivsc.max())
plt.xlim(-1, min(1000, len(z_scores_embryo_ivsc)))
plt.ylabel("embryo_ivsc z-score")
plt.title("IVSC SMD z-scores")
plt.tight_layout()
plt.savefig(fig_path / "ivsc_smd_z_scores.png", dpi=300, bbox_inches="tight")
plt.savefig(fig_path / "ivsc_smd_z_scores.pdf", bbox_inches="tight")
plt.show()
print(f"{(z_scores_embryo_ivsc > smd_cutoff).sum()} IVSC genes with z_score >= cutoff")


## Select IVSC SMD Genes

In [ ]:

genes_cellcycle = [
    "MKI67", "CENPE", "SGO2", "KIF14", "PIF1", "NDC80", "CDCA8", "PLK1", "AURKA", "UBE2C",
    "ASPM", "TOP2A", "TPX2", "NUSAP1", "CDC20", "CKS2", "KPNA2", "TUBB4B", "DLGAP5", "BIRC5",
    "HMMR", "CCNB1", "ARL6IP1", "PTTG1", "UBE2S", "DUT", "HELLS", "CLSPN", "RRM2", "PCLAF",
    "TYMS", "KIF18B", "SMC4", "HIST1H4C", "DIAPH3", "RFC3", "MIR924HG", "MIS18BP1", "TUBA1C", "CDK1",
    "CENPA", "KIF11", "ECT2", "KNL1", "NEK2", "CEP55", "PSRC1", "CDCA3", "CCNA2", "GTSE1",
    "CENPF", "CKS1B", "MELK", "CDCA5", "MCM4", "HIST1H1E", "BRCA2", "TRIM66", "C1orf112", "POLD3",
    "KIF23", "CDKAL1", "RAD51AP1", "RANBP1", "CDCA2", "CAND2", "EIF5A", "SRSF2", "GINS1", "GINS2",
]

manual_addgenes = [
    "FOXA2", "SHH", "ARX", "CORIN", "NKX6-1", "NKX6-2", "SOX2", "FOXA1", "TFF3", "PKD1L1", "NOTO", "PTCH1", "FERD3L",
]
manual_removegenes = []

cellcycle_gene_ids = set(
    resolve_symbols(adata_embryo_ivsc_, genes_cellcycle, strict=False, allow_missing=True)
)
SMDembryo_ivscgenes = list(
    adata_embryo_ivsc_filtered.var_names[
        adata_embryo_ivsc_filtered.var.z_score_embryo_ivsc > smd_cutoff
    ]
)
for gene_symbol in manual_addgenes:
    for gene_id in resolve_symbols(adata_embryo_ivsc_, [gene_symbol], strict=False, allow_missing=True):
        if gene_id not in SMDembryo_ivscgenes:
            SMDembryo_ivscgenes.append(gene_id)
for gene_symbol in manual_removegenes:
    for gene_id in resolve_symbols(adata_embryo_ivsc_, [gene_symbol], strict=False, allow_missing=True):
        if gene_id in SMDembryo_ivscgenes:
            SMDembryo_ivscgenes.remove(gene_id)
SMDembryo_ivscgenes = [g for g in SMDembryo_ivscgenes if g not in cellcycle_gene_ids]

adata_embryo_ivsc_SMD = adata_embryo_ivsc_[:, SMDembryo_ivscgenes].copy()
print(f"{adata_embryo_ivsc_SMD.n_vars} IVSC genes selected before correlation refinement")


In [ ]:

corr_embryo_ivsc_gg = gene_corrcoef_sparse_safe(adata_embryo_ivsc_SMD.X)
keep_gene_ids = []
for i in range(corr_embryo_ivsc_gg.shape[0]):
    off_diag = np.delete(corr_embryo_ivsc_gg[i], i)
    l2_corr = np.sqrt(np.sum(np.square(off_diag)) / np.size(off_diag))
    if l2_corr > 0.06:
        keep_gene_ids.append(adata_embryo_ivsc_SMD.var_names[i])

for gene_id in resolve_symbols(adata_embryo_ivsc_, manual_addgenes, strict=False, allow_missing=True):
    if gene_id not in keep_gene_ids:
        keep_gene_ids.append(gene_id)
keep_gene_ids = [g for g in list(dict.fromkeys(keep_gene_ids)) if g not in cellcycle_gene_ids]

adata_embryo_ivsc_SMD = adata_embryo_ivsc_[:, keep_gene_ids].copy()
selected_smd_gene_table = pd.DataFrame(
    {
        "gene_id": adata_embryo_ivsc_SMD.var_names,
        "gene_symbol": adata_embryo_ivsc_SMD.var["gene_symbol"].astype(str).values,
        "z_score_embryo_ivsc": adata_embryo_ivsc_SMD.var["z_score_embryo_ivsc"].astype(float).values,
    }
).sort_values(["z_score_embryo_ivsc", "gene_symbol"], ascending=[False, True])
selected_smd_gene_table.to_csv(table_path / "ivsc_selected_smd_genes.csv", index=False)
print(f"{adata_embryo_ivsc_SMD.n_vars} IVSC genes retained after correlation refinement")
selected_smd_gene_table.head(40)


In [ ]:

corr_embryo_ivsc_gg = gene_corrcoef_sparse_safe(adata_embryo_ivsc_SMD.X)
cluster_gg_embryo_ivsc = sns.clustermap(
    corr_embryo_ivsc_gg,
    method="ward",
    metric="euclidean",
    figsize=(12, 12),
    cmap=batlow,
    vmin=-0.2,
    vmax=0.8,
    yticklabels=var_names_to_symbols(adata_embryo_ivsc_SMD),
    xticklabels=var_names_to_symbols(adata_embryo_ivsc_SMD),
)
cluster_gg_embryo_ivsc.savefig(fig_path / "ivsc_smd_gene_gene_correlation.png", dpi=300)
cluster_gg_embryo_ivsc.savefig(fig_path / "ivsc_smd_gene_gene_correlation.pdf")
plt.show()


## Screen a Small Leiden Resolution Ladder

In [ ]:
scan_resolutions = [0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20]
resolution_scan_rows = []
scan_marker_symbols = FP_POSITIVE_MARKERS + FP_CONTEXT_MARKERS + ["NOTO", "PKD1L1"]
scan_marker_cache = {symbol: expr_vector_by_symbol(adata_embryo_ivsc_, symbol) for symbol in scan_marker_symbols}

adata_embryo_ivsc_SMD_scan = adata_embryo_ivsc_SMD.copy()
sc.pp.normalize_total(adata_embryo_ivsc_SMD_scan)
sc.pp.neighbors(adata_embryo_ivsc_SMD_scan, use_rep="X")

for resolution in scan_resolutions:
    adata_scan = adata_embryo_ivsc_SMD_scan.copy()
    sc.tl.leiden(
        adata_scan,
        resolution=resolution,
        random_state=0,
        key_added="leiden_embryo_ivsc_scan",
        flavor="igraph",
        n_iterations=-1,
    )
    adata_scan_expr = adata_embryo_ivsc_.copy()
    adata_scan_expr.obs["leiden_embryo_ivsc_scan"] = adata_scan.obs["leiden_embryo_ivsc_scan"].astype("category")
    marker_summary_df = cluster_marker_summary(
        adata_expr=adata_scan_expr,
        cluster_key="leiden_embryo_ivsc_scan",
        marker_symbols=scan_marker_symbols,
        marker_cache=scan_marker_cache,
    )
    marker_summary_df["resolution"] = resolution
    marker_summary_df["n_clusters"] = int(adata_scan.obs["leiden_embryo_ivsc_scan"].nunique())
    marker_summary_df["fp_score"] = floor_plate_score(marker_summary_df)
    resolution_scan_rows.append(marker_summary_df)

resolution_scan_df = pd.concat(resolution_scan_rows, ignore_index=True)
resolution_scan_df.to_csv(table_path / "ivsc_resolution_marker_scan.csv", index=False)
resolution_scan_best = (
    resolution_scan_df.sort_values(["resolution", "fp_score"], ascending=[True, False])
    .groupby("resolution")
    .head(1)
    .reset_index(drop=True)
)
resolution_scan_best.to_csv(table_path / "ivsc_resolution_best_clusters.csv", index=False)
resolution_scan_best[
    [
        "resolution", "n_clusters", "cluster", "n_cells", "fp_score",
        "FOXA2_mean", "SHH_mean", "ARX_mean", "CORIN_mean", "PTCH1_mean", "SOX2_mean", "NOTO_mean", "PKD1L1_mean",
    ]
]


## Cluster IVSC in SMD Gene Space

In [ ]:
CHOSEN_LEIDEN_RESOLUTION = 0.10
CHOSEN_LEIDEN_REASON = (
    "First raw Leiden resolution on the default IVSC SMD neighbor graph that isolates a single "
    "strong FOXA2/SHH/ARX/CORIN-positive floor-plate-like cluster; that raw cluster is then "
    "collapsed against the remainder of IVSC to match the biological question."
)

adata_embryo_ivsc_SMD_ = adata_embryo_ivsc_SMD.copy()
sc.pp.normalize_total(adata_embryo_ivsc_SMD_)
sc.pp.neighbors(adata_embryo_ivsc_SMD_, use_rep="X")
sc.tl.umap(adata_embryo_ivsc_SMD_, random_state=0)
sc.tl.leiden(
    adata_embryo_ivsc_SMD_,
    resolution=CHOSEN_LEIDEN_RESOLUTION,
    random_state=0,
    key_added="leiden_embryo_ivsc_raw",
    flavor="igraph",
    n_iterations=-1,
)

adata_embryo_ivsc_with_clusters = adata_embryo_ivsc_.copy()
adata_embryo_ivsc_with_clusters.obs["leiden_embryo_ivsc_raw"] = adata_embryo_ivsc_SMD_.obs["leiden_embryo_ivsc_raw"].astype("category")

cluster_marker_symbols = FP_POSITIVE_MARKERS + FP_CONTEXT_MARKERS + ["NOTO", "PKD1L1"]
cluster_marker_cache = {symbol: expr_vector_by_symbol(adata_embryo_ivsc_with_clusters, symbol) for symbol in cluster_marker_symbols}

raw_cluster_marker_summary = cluster_marker_summary(
    adata_expr=adata_embryo_ivsc_with_clusters,
    cluster_key="leiden_embryo_ivsc_raw",
    marker_symbols=cluster_marker_symbols,
    marker_cache=cluster_marker_cache,
)
raw_cluster_marker_summary["fp_score"] = floor_plate_score(raw_cluster_marker_summary)
raw_cluster_marker_summary = raw_cluster_marker_summary.sort_values(["fp_score", "n_cells"], ascending=[False, False]).reset_index(drop=True)
raw_cluster_marker_summary.to_csv(table_path / "ivsc_raw_cluster_marker_summary.csv", index=False)

floor_plate_raw_cluster = str(raw_cluster_marker_summary.iloc[0]["cluster"])

collapsed_labels = np.where(
    adata_embryo_ivsc_SMD_.obs["leiden_embryo_ivsc_raw"].astype(str).values == floor_plate_raw_cluster,
    "Floor Plate-like",
    "Other Intermediate-Ventral Spinal Cord",
)
adata_embryo_ivsc_SMD_.obs["leiden_embryo_ivsc"] = pd.Categorical(
    collapsed_labels,
    categories=["Floor Plate-like", "Other Intermediate-Ventral Spinal Cord"],
    ordered=True,
)
adata_embryo_ivsc_with_clusters.obs["leiden_embryo_ivsc"] = adata_embryo_ivsc_SMD_.obs["leiden_embryo_ivsc"]

chosen_cluster_marker_summary = cluster_marker_summary(
    adata_expr=adata_embryo_ivsc_with_clusters,
    cluster_key="leiden_embryo_ivsc",
    marker_symbols=cluster_marker_symbols,
    marker_cache=cluster_marker_cache,
)
chosen_cluster_marker_summary["fp_score"] = floor_plate_score(chosen_cluster_marker_summary)
chosen_cluster_marker_summary.to_csv(table_path / "ivsc_chosen_cluster_marker_summary.csv", index=False)

print(f"Chosen raw IVSC floor-plate cluster: {floor_plate_raw_cluster}")
raw_cluster_marker_summary[
    [
        "cluster", "n_cells", "fp_score",
        "FOXA2_mean", "SHH_mean", "ARX_mean", "CORIN_mean", "PTCH1_mean", "SOX2_mean", "NOTO_mean", "PKD1L1_mean",
    ]
]


In [ ]:

adata_embryo_ivsc_SMD_leiden = sc.get.aggregate(
    adata_embryo_ivsc_SMD_,
    by="leiden_embryo_ivsc",
    func=["count_nonzero", "mean", "sum", "var"],
    axis="obs",
)

corr_clcl_embryo_ivsc = np.corrcoef(adata_embryo_ivsc_SMD_leiden.layers["mean"])
with plt.rc_context({"figure.figsize": (6, 5), "figure.dpi": 300}):
    cluster_clcl_embryo_ivsc = sns.clustermap(
        corr_clcl_embryo_ivsc,
        method="ward",
        metric="euclidean",
        figsize=(6, 6),
        cmap=batlow,
        yticklabels=adata_embryo_ivsc_SMD_leiden.obs.leiden_embryo_ivsc.cat.categories,
        xticklabels=adata_embryo_ivsc_SMD_leiden.obs.leiden_embryo_ivsc.cat.categories,
    )
    cluster_clcl_embryo_ivsc.fig.suptitle(
        "Cluster-cluster mean gene correlation among selected IVSC SMD genes (collapsed floor-plate separation)"
    )
    cluster_clcl_embryo_ivsc.savefig(fig_path / "ivsc_cluster_cluster_correlation.png", dpi=300)
    cluster_clcl_embryo_ivsc.savefig(fig_path / "ivsc_cluster_cluster_correlation.pdf")
plt.show()


In [ ]:
adata_embryo_ivsc_SMD_ = adata_embryo_ivsc_SMD_[
    adata_embryo_ivsc_SMD_.obs.sort_values(["leiden_embryo_ivsc", "leiden_embryo_ivsc_raw"]).index, :
]
adata_embryo_ivsc_with_clusters = adata_embryo_ivsc_with_clusters[
    adata_embryo_ivsc_with_clusters.obs.sort_values(["leiden_embryo_ivsc", "leiden_embryo_ivsc_raw"]).index, :
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=300)
sc.pl.umap(
    adata_embryo_ivsc_SMD_,
    color="leiden_embryo_ivsc_raw",
    title=f"Week 4 IVSC raw Leiden (res={CHOSEN_LEIDEN_RESOLUTION:.2f})",
    size=80,
    show=False,
    ax=axes[0],
)
sc.pl.umap(
    adata_embryo_ivsc_SMD_,
    color="leiden_embryo_ivsc",
    title="Week 4 IVSC floor plate separation",
    size=80,
    palette=["#d95f02", "#1b9e77"],
    show=False,
    ax=axes[1],
)
fig.tight_layout()
fig.savefig(fig_path / "ivsc_umap.png", dpi=300, bbox_inches="tight")
fig.savefig(fig_path / "ivsc_umap.pdf", bbox_inches="tight")
plt.show()


## Calculate IVSC DEGs by Multinomial Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_cluster_order = list(adata_embryo_ivsc_with_clusters.obs["leiden_embryo_ivsc"].cat.categories)
X_logreg = adata_embryo_ivsc_with_clusters.X
if scipy.sparse.issparse(X_logreg):
    X_logreg = X_logreg.tocsr()
y_codes = adata_embryo_ivsc_with_clusters.obs["leiden_embryo_ivsc"].cat.codes.to_numpy()

clf = LogisticRegression(
    max_iter=1000,
    multi_class="multinomial",
    solver="lbfgs",
    random_state=0,
)
clf.fit(X_logreg, y_codes)

coef_map = {}
if clf.coef_.shape[0] == 1 and len(logreg_cluster_order) == 2:
    positive_code = int(clf.classes_[1])
    negative_code = int(clf.classes_[0])
    coef_map[logreg_cluster_order[positive_code]] = clf.coef_[0]
    coef_map[logreg_cluster_order[negative_code]] = -clf.coef_[0]
else:
    for row_idx, class_code in enumerate(clf.classes_):
        coef_map[logreg_cluster_order[int(class_code)]] = clf.coef_[row_idx]

deg_name_map = {}
deg_score_map = {}
for cluster_name in logreg_cluster_order:
    coef_vector = coef_map[cluster_name]
    ranked_idx = np.argsort(coef_vector)[::-1][:100]
    deg_name_map[cluster_name] = adata_embryo_ivsc_with_clusters.var_names[ranked_idx].tolist()
    deg_score_map[cluster_name] = coef_vector[ranked_idx].astype(float).tolist()

DEGs_embryo_multi_ivsc = pd.DataFrame(deg_name_map)
DEGscores_embryo_multi_ivsc = pd.DataFrame(deg_score_map)
DEGs_embryo_multi_ivsc.to_csv(table_path / "ivsc_top100_logreg_deg_names.csv", index=False)
DEGscores_embryo_multi_ivsc.to_csv(table_path / "ivsc_top100_logreg_deg_scores.csv", index=False)

deg_long_rows = []
for cluster_name in DEGs_embryo_multi_ivsc.columns:
    for rank, (gene_id, score) in enumerate(
        zip(DEGs_embryo_multi_ivsc[cluster_name], DEGscores_embryo_multi_ivsc[cluster_name]),
        start=1,
    ):
        deg_long_rows.append(
            {
                "cluster": cluster_name,
                "rank": rank,
                "gene_id": gene_id,
                "gene_symbol": str(adata_embryo_ivsc_with_clusters.var.loc[gene_id, "gene_symbol"]),
                "score": float(score),
            }
        )
deg_long_df = pd.DataFrame(deg_long_rows)
deg_long_df.to_csv(table_path / "ivsc_top100_logreg_deg_long.csv", index=False)

deg_long_df.head(20)


## IVSC DEG Heatmap with Intracluster Cell Ordering

In [ ]:
top_deg_per_cluster_for_heatmap = 20
heatmap_gene_ids = []
heatmap_cluster_order = [
    cluster_name
    for cluster_name in adata_embryo_ivsc_with_clusters.obs["leiden_embryo_ivsc"].cat.categories
    if cluster_name in DEGs_embryo_multi_ivsc.columns
]
for cluster_name in heatmap_cluster_order:
    cluster_gene_ids = DEGs_embryo_multi_ivsc[cluster_name].tolist()[:top_deg_per_cluster_for_heatmap]
    for gene_id in cluster_gene_ids:
        if gene_id not in heatmap_gene_ids:
            heatmap_gene_ids.append(gene_id)

adata_embryo_ivsc_heatmap = adata_embryo_ivsc_with_clusters[:, heatmap_gene_ids].copy()
heatmap_cell_order = ordered_cells_by_cluster_clustermap(
    adata=adata_embryo_ivsc_heatmap,
    cluster_key="leiden_embryo_ivsc",
    var_names=adata_embryo_ivsc_heatmap.var_names,
    figsize=(6, 6),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="ward",
    metric="euclidean",
)
pd.DataFrame(
    {
        "cell_id": heatmap_cell_order,
        "leiden_embryo_ivsc": adata_embryo_ivsc_with_clusters.obs.loc[heatmap_cell_order, "leiden_embryo_ivsc"].astype(str).values,
    }
).to_csv(table_path / "ivsc_heatmap_cell_order.csv", index=False)

adata_embryo_ivsc_heatmap = adata_embryo_ivsc_heatmap[heatmap_cell_order, heatmap_gene_ids].copy()

with plt.rc_context({"figure.dpi": 300}):
    sc.pl.heatmap(
        adata_embryo_ivsc_heatmap,
        var_names=heatmap_gene_ids,
        groupby="leiden_embryo_ivsc",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(8, 6),
        cmap=batlow,
        vmin=0,
        vmax=5,
        dendrogram=False,
        show=False,
    )
    plt.savefig(fig_path / "ivsc_logreg_deg_heatmap.png", dpi=300, bbox_inches="tight")
    plt.savefig(fig_path / "ivsc_logreg_deg_heatmap.pdf", bbox_inches="tight")
    plt.show()


## Save Stage Outputs

In [ ]:
save_npz(stage_path / "embryo_ivsc_07b.npz", adata_embryo_ivsc_filtered.X)
save_h5ad(adata_embryo_ivsc_filtered, stage_path / "adata_embryo_ivsc_filtered.h5ad")
save_h5ad(adata_embryo_ivsc_, stage_path / "adata_embryo_ivsc_.h5ad")
save_h5ad(adata_embryo_ivsc_SMD, stage_path / "adata_embryo_ivsc_SMD.h5ad")
save_h5ad(adata_embryo_ivsc_SMD_, stage_path / "adata_embryo_ivsc_SMD_.h5ad")
save_h5ad(adata_embryo_ivsc_with_clusters, stage_path / "adata_embryo_ivsc_with_clusters.h5ad")

adata_embryo_ivsc_with_clusters.obs[["leiden_embryo", "leiden_embryo_ivsc", "leiden_embryo_ivsc_raw"]].to_csv(
    table_path / "ivsc_cluster_labels.csv"
)

save_json(
    {
        "stage": STAGE_ID,
        "seed_policy": "all seeds set to 0",
        "upstream_stage": "07_human_embryo",
        "parent_clusters": ivsc_parent_clusters,
        "preferred_fresh_smd_substrate": "embryo_ivsc",
        "n_parent_cluster_cells": int(adata_embryo_ivsc_.n_obs),
        "n_pre_smd_genes": int(adata_embryo_ivsc_filtered.n_vars),
        "n_postfilter_genes": int(adata_embryo_ivsc_.n_vars),
        "recommended_n_sub": int(0.8 * adata_embryo_ivsc_filtered.n_obs),
        "smd_input_matrix": "embryo_ivsc_07b.npz",
        "filtered_artifact": "adata_embryo_ivsc_filtered.h5ad",
        "downstream_artifact": "adata_embryo_ivsc_.h5ad",
        "smd_artifact": "adata_embryo_ivsc_SMD.h5ad",
        "smd_cluster_artifact": "adata_embryo_ivsc_with_clusters.h5ad",
        "smd_z_score_file": z_score_file.name,
        "smd_cutoff": smd_cutoff,
        "n_selected_smd_genes": int(adata_embryo_ivsc_SMD.n_vars),
        "resolution_scan_values": scan_resolutions,
        "neighbor_graph_method": "scanpy default neighbors on selected IVSC SMD genes",
        "chosen_leiden_resolution": CHOSEN_LEIDEN_RESOLUTION,
        "chosen_leiden_reason": CHOSEN_LEIDEN_REASON,
        "raw_cluster_key": "leiden_embryo_ivsc_raw",
        "raw_cluster_labels": list(adata_embryo_ivsc_with_clusters.obs["leiden_embryo_ivsc_raw"].cat.categories),
        "floor_plate_raw_cluster": floor_plate_raw_cluster,
        "cluster_key": "leiden_embryo_ivsc",
        "cluster_labels": list(adata_embryo_ivsc_with_clusters.obs["leiden_embryo_ivsc"].cat.categories),
        "floor_plate_cluster_name": "Floor Plate-like",
        "other_cluster_name": "Other Intermediate-Ventral Spinal Cord",
        "tables_dir": "tables",
        "figures_dir": "figures",
    },
    stage_path / "meta.json",
)

print(f"Saved IVSC clustering outputs to {stage_path}")
